# LipiLens - Colab inference server

Run every cell **in order**. Takes ~4 minutes to load the model.

**Step 1 of 2:** open this notebook in Colab with a **GPU runtime**

`Runtime` -> `Change runtime type` -> `T4 GPU` -> `Save`

**Step 2 of 2:** when the tunnel URL is printed at the end, paste it into the
`.env` file in your project folder on your own machine.

In [ ]:
# Cell 1 - install dependencies (pinned to match the local environment)
!pip install -q "transformers==5.17.0" "peft==0.21.0" "bitsandbytes==0.50.2" \
              "accelerate==1.15.0" pillow fastapi uvicorn pyngrok requests

In [ ]:
# Cell 2 - confirm the GPU is actually there
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_info(0).total_memory / 1024**3, 1), "GB")
print("torch:", torch.__version__)

## Upload the server

Click the folder icon on the left, then the upload icon, and upload
`colab/lipilens_colab_server.py` from your project.

In [ ]:
# Cell 3 - start the API in a background thread
# (a plain `!python server.py` would block this cell forever)
import threading, uvicorn, importlib.util, sys

spec = importlib.util.spec_from_file_location("lipilens_colab_server", "lipilens_colab_server.py")
server = importlib.util.module_from_spec(spec)
sys.modules["lipilens_colab_server"] = server
spec.loader.exec_module(server)          # __name__ != "__main__", so it does not self-start

threading.Thread(
    target=lambda: uvicorn.run(server.app, host="0.0.0.0", port=8000, log_level="info"),
    daemon=True,
).start()

import time, requests
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            print("server up on port 8000")
            break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("server did not start - check the output above")

## Expose it to your laptop

**Option A - ngrok (recommended).** Create a free authtoken at
<https://dashboard.ngrok.com/get-started/your-authtoken> and paste it below.

**Option B - no signup.** Skip to the next-but-one cell and use Colab's
built-in tunnel instead.

In [ ]:
# Cell 4 - ngrok tunnel
from pyngrok import ngrok

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"   # <- edit this line

if NGROK_TOKEN.startswith("PASTE"):
    print("No token pasted - skipping ngrok. Use Option B instead.")
    TUNNEL_URL = None
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    TUNNEL_URL = ngrok.connect(8000).public_url
    print("COLAB_ENDPOINT_URL=" + TUNNEL_URL)

In [ ]:
# Cell 4b - Option B: Colab's built-in tunnel (no signup needed)
# Run this INSTEAD of cell 4 if you do not want an ngrok account.
from google.colab import output
output.serve_kernel_port_as_window(8000)
TUNNEL_URL = "PASTE_THE_URL_FROM_THE_POPUP_ABOVE_HERE"

In [ ]:
# Cell 5 - self-test: prove the tunnel + model work BEFORE you go to your laptop
# This loads the model (~1-3 min) and transcribes one small line image.
import base64, io, requests, numpy as np
from PIL import Image, ImageDraw

if not TUNNEL_URL or TUNNEL_URL.startswith("PASTE"):
    raise SystemExit("Set TUNNEL_URL first (cell 4 or 4b)")

print("remote /gpu ->", requests.get(TUNNEL_URL + "/gpu", timeout=30).json())

img = Image.new("RGB", (700, 90), "white")
ImageDraw.Draw(img).text((12, 30), "test line 12345", fill="black")
buf = io.BytesIO()
img.save(buf, format="PNG")

r = requests.post(
    TUNNEL_URL + "/transcribe",
    json={"image": base64.b64encode(buf.getvalue()).decode(),
          "prompt": "Transliterate the text in this image into Devanagari script. Output only the Devanagari text."},
    timeout=900,
)
print("status:", r.status_code)
print("result:", r.json())

if r.status_code == 200:
    print("\nSERVER IS WORKING. Copy the printed URL into your .env.")
else:
    print("\nSomething failed - read the error above before continuing.")

## On your own machine

Open `.env` in the project root and set:

```
INFERENCE_MODE=colab
COLAB_ENDPOINT_URL=<the URL printed above>
```

Then start the app:

```bash
python -m uvicorn backend.main:app --port 8000
cd frontend && npm run dev
```

Open <http://localhost:8000/api/health> - it should say `"status": "ok"`
instead of `"degraded"`.

**Keep this tab open.** Closing it or letting the runtime time out kills the
server and the app goes back to degraded.